In [ ]:
import math


def caesar_encrypt(plaintext: str, shift: int) -> str:
    result = []
    for char in plaintext:
        if char.isalpha():
            base = ord("A") if char.isupper() else ord("a")
            result.append(chr((ord(char) - base + shift) % 26 + base))
        else:
            result.append(char)
    return "".join(result)


def caesar_decrypt(ciphertext: str, shift: int) -> str:
    return caesar_encrypt(ciphertext, -shift)


def generate_playfair_matrix(key: str) -> list[list[str]]:
    key = key.upper().replace("J", "I")
    seen = []
    for char in key + "ABCDEFGHIKLMNOPQRSTUVWXYZ":
        if char.isalpha() and char not in seen:
            seen.append(char)
    return [seen[i : i + 5] for i in range(0, 25, 5)]


def find_playfair_pos(matrix: list[list[str]], char: str) -> tuple[int, int]:
    for r in range(5):
        for c in range(5):
            if matrix[r][c] == char:
                return r, c
    return -1, -1


def prepare_playfair_text(text: str) -> str:
    text = "".join(c for c in text.upper() if c.isalpha()).replace("J", "I")
    prepared = []
    i = 0
    while i < len(text):
        a = text[i]
        b = text[i + 1] if (i + 1) < len(text) else "X"
        if a == b:
            prepared.append(a + "X")
            i += 1
        else:
            prepared.append(a + b)
            i += 2
    return "".join(prepared)


def playfair_process(text: str, key: str, encrypt: bool = True) -> str:
    matrix = generate_playfair_matrix(key)
    if encrypt:
        text = prepare_playfair_text(text)
    shift = 1 if encrypt else -1

    result = []
    for i in range(0, len(text), 2):
        r1, c1 = find_playfair_pos(matrix, text[i])
        r2, c2 = find_playfair_pos(matrix, text[i + 1])
        if r1 == r2:
            result.append(matrix[r1][(c1 + shift) % 5] + matrix[r2][(c2 + shift) % 5])
        elif c1 == c2:
            result.append(matrix[(r1 + shift) % 5][c1] + matrix[(r2 + shift) % 5][c2])
        else:
            result.append(matrix[r1][c2] + matrix[r2][c1])
    return "".join(result)


def vigenere_process(text: str, key: str, encrypt: bool = True) -> str:
    clean_key = "".join(c for c in key.upper() if c.isalpha())
    result = []
    key_idx = 0

    for char in text:
        if char.isalpha():
            base = ord("A") if char.isupper() else ord("a")
            shift = ord(clean_key[key_idx % len(clean_key)]) - ord("A")
            if not encrypt:
                shift = -shift
            result.append(chr((ord(char) - base + shift) % 26 + base))
            key_idx += 1
        else:
            result.append(char)
    return "".join(result)


def otp_alpha_process(text: str, key: str, encrypt: bool = True) -> str:
    clean_text = "".join(c for c in text.upper() if c.isalpha())
    clean_key = "".join(c for c in key.upper() if c.isalpha())
    if len(clean_key) < len(clean_text):
        raise ValueError("OTP key must be at least as long as the text.")

    result = []
    for t, k in zip(clean_text, clean_key):
        t_val = ord(t) - ord("A")
        k_val = ord(k) - ord("A")
        val = (t_val + k_val) % 26 if encrypt else (t_val - k_val) % 26
        result.append(chr(ord("A") + val))
    return "".join(result)


def otp_xor_process(text: str, key: str) -> str:
    if len(key) < len(text):
        raise ValueError("OTP key must be at least as long as the text.")
    return "".join(chr(ord(t) ^ ord(k)) for t, k in zip(text, key))


def rail_fence_encrypt(plaintext: str, rails: int) -> str:
    if rails <= 1:
        return plaintext
    fence = [[] for _ in range(rails)]
    rail, direction = 0, 1

    for char in plaintext:
        fence[rail].append(char)
        rail += direction
        if rail == 0 or rail == rails - 1:
            direction *= -1

    return "".join("".join(row) for row in fence)


def rail_fence_decrypt(ciphertext: str, rails: int) -> str:
    if rails <= 1:
        return ciphertext
    n = len(ciphertext)
    pattern = [0] * n
    rail, direction = 0, 1

    for i in range(n):
        pattern[i] = rail
        rail += direction
        if rail == 0 or rail == rails - 1:
            direction *= -1

    fence = [[] for _ in range(rails)]
    idx = 0
    for r in range(rails):
        for i in range(n):
            if pattern[i] == r:
                fence[r].append(ciphertext[idx])
                idx += 1

    result = []
    rail_pointers = [0] * rails
    for i in range(n):
        r = pattern[i]
        result.append(fence[r][rail_pointers[r]])
        rail_pointers[r] += 1
    return "".join(result)


def get_column_order(key: str) -> list[int]:
    return [idx for idx, _ in sorted(enumerate(key), key=lambda x: (x[1], x[0]))]


def columnar_encrypt(plaintext: str, key: str, pad_char: str = "X") -> str:
    cols = len(key)
    rows = math.ceil(len(plaintext) / cols)
    padded = plaintext.ljust(rows * cols, pad_char)

    grid = [list(padded[i : i + cols]) for i in range(0, len(padded), cols)]
    col_order = get_column_order(key)

    return "".join(grid[r][c] for c in col_order for r in range(rows))


def columnar_decrypt(ciphertext: str, key: str) -> str:
    cols = len(key)
    rows = math.ceil(len(ciphertext) / cols)
    grid = [[""] * cols for _ in range(rows)]
    col_order = get_column_order(key)

    idx = 0
    for c in col_order:
        for r in range(rows):
            if idx < len(ciphertext):
                grid[r][c] = ciphertext[idx]
                idx += 1

    return "".join("".join(row) for row in grid)


def mod_inverse(a: int, m: int = 26) -> int:
    a = a % m
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    raise ValueError("Key matrix is not invertible modulo 26.")


def inverse_matrix_2x2(m: list[list[int]]) -> list[list[int]]:
    det = m[0][0] * m[1][1] - m[0][1] * m[1][0]
    det_inv = mod_inverse(det, 26)
    return [
        [(det_inv * m[1][1]) % 26, (det_inv * -m[0][1]) % 26],
        [(det_inv * -m[1][0]) % 26, (det_inv * m[0][0]) % 26],
    ]


def inverse_matrix_3x3(m: list[list[int]]) -> list[list[int]]:
    det = (
        m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
        - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
        + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0])
    )
    det_inv = mod_inverse(det, 26)
    inv = [[0] * 3 for _ in range(3)]

    for r in range(3):
        for c in range(3):
            r1, r2 = (r + 1) % 3, (r + 2) % 3
            c1, c2 = (c + 1) % 3, (c + 2) % 3
            cofactor = m[c1][r1] * m[c2][r2] - m[c1][r2] * m[c2][r1]
            inv[r][c] = (det_inv * cofactor) % 26
    return inv


def hill_multiply_blocks(text: str, matrix: list[list[int]]) -> str:
    n = len(matrix)
    clean = "".join(c for c in text.upper() if c.isalpha())
    while len(clean) % n != 0:
        clean += "X"

    result = []
    for i in range(0, len(clean), n):
        for r in range(n):
            total = sum(matrix[r][c] * (ord(clean[i + c]) - ord("A")) for c in range(n))
            result.append(chr(ord("A") + (total % 26)))
    return "".join(result)


def hill_encrypt(plaintext: str, key_matrix: list[list[int]]) -> str:
    return hill_multiply_blocks(plaintext, key_matrix)


def hill_decrypt(ciphertext: str, key_matrix: list[list[int]]) -> str:
    n = len(key_matrix)
    inv_matrix = inverse_matrix_2x2(key_matrix) if n == 2 else inverse_matrix_3x3(key_matrix)
    return hill_multiply_blocks(ciphertext, inv_matrix)


if __name__ == "__main__":
    c_enc = caesar_encrypt("HELLO WORLD", 3)
    print(f"1. Caesar     -> Enc: {c_enc} | Dec: {caesar_decrypt(c_enc, 3)}")

    p_enc = playfair_process("INSTRUMENTS", "MONARCHY", encrypt=True)
    print(f"2. Playfair   -> Enc: {p_enc} | Dec: {playfair_process(p_enc, 'MONARCHY', encrypt=False)}")

    v_enc = vigenere_process("ATTAC", "LEMON", encrypt=True)
    print(f"3. Vigenère   -> Enc: {v_enc} | Dec: {vigenere_process(v_enc, 'LEMON', encrypt=False)}")

    o_enc = otp_alpha_process("HELLO", "XMCKL", encrypt=True)
    print(f"4. OTP (Mod26)-> Enc: {o_enc} | Dec: {otp_alpha_process(o_enc, 'XMCKL', encrypt=False)}")

    r_enc = rail_fence_encrypt("WEAREDISCOVEREDFLEEATONCE", 3)
    print(f"5. Rail Fence -> Enc: {r_enc} | Dec: {rail_fence_decrypt(r_enc, 3)}")

    col_enc = columnar_encrypt("WEAREDISCOVEREDFLEEATONCE", "ZEBRA")
    print(f"6. Columnar   -> Enc: {col_enc} | Dec: {columnar_decrypt(col_enc, 'ZEBRA')}")

    hill_key = [[3, 3], [2, 5]]
    h_enc = hill_encrypt("HELP", hill_key)
    print(f"7. Hill       -> Enc: {h_enc} | Dec: {hill_decrypt(h_enc, hill_key)}")